# DS2 - Matrix Addition using CUDA Python (Numba)

## Dataset Metadata

- Dataset Name: Matrix Dataset
- Source: Kaggle
- Kaggle Dataset: https://www.kaggle.com/datasets/julianezrasamuel/matrix-dataset
- File Type: Pickle (`.pkl`)
- Matrix Size: 1024 x 1024
- Data Type: `float32` / `float64`
- Number of Matrices: Multiple matrices available
- Total Elements per Matrix: 1,048,576
- CUDA Operation: Matrix Addition

## CUDA Task

Input:
- Matrix `A` of shape `1024 x 1024`
- Matrix `B` of shape `1024 x 1024`

Output:
- Matrix `C = A + B`

Parallelization Strategy:
- One CUDA thread computes one matrix element.

Important for Kaggle:
- Add the Matrix Dataset using Kaggle's "Add Data" panel.
- This notebook does not download the dataset.
- It only reads files from `/kaggle/input` and saves optional outputs to `/kaggle/working`.

In [7]:
# Standard libraries used for loading the Kaggle dataset and measuring execution time.
from pathlib import Path
import pickle
import time

# Numerical, dataframe, and GPU libraries available in Kaggle notebooks.
import numpy as np
import pandas as pd
from numba import cuda


# Kaggle automatically mounts added datasets here.
INPUT_ROOT = Path("/kaggle/input")

# Kaggle allows notebook outputs to be written here.
WORKING_ROOT = Path("/kaggle/working")

# Dataset metadata used in error messages and file discovery.
DATASET_URL = "https://www.kaggle.com/datasets/julianezrasamuel/matrix-dataset"
DATASET_SLUG_HINT = "matrix-dataset"
MATRIX_SIZE = 1024
VALUES_PER_MATRIX = MATRIX_SIZE * MATRIX_SIZE
VALUES_FOR_TWO_MATRICES = VALUES_PER_MATRIX * 2


def find_matrix_data_files(root=INPUT_ROOT):
    """Find Matrix Dataset files from Kaggle input without downloading anything."""
    if not root.exists():
        raise FileNotFoundError(
            "Kaggle input directory was not found. Add the Matrix Dataset in Kaggle: "
            f"{DATASET_URL}"
        )

    # The dataset is documented as .pkl, but this also accepts related binary formats.
    patterns = ["*.pkl", "*.pickle", "*.joblib", "*.npy", "*.npz", "*.parquet", "*.feather", "*.h5", "*.hdf5"]
    all_files = []
    for pattern in patterns:
        all_files.extend(root.rglob(pattern))

    all_files = sorted(set(all_files), key=lambda p: p.stat().st_size, reverse=True)

    # Prefer the Matrix Dataset folder, but keep a fallback for Kaggle mount-name changes.
    preferred_files = [
        p for p in all_files
        if DATASET_SLUG_HINT in str(p).lower() or "matrix" in str(p).lower()
    ]
    data_files = preferred_files or all_files

    if not data_files:
        raise FileNotFoundError(
            "No supported Matrix Dataset file was found under /kaggle/input. "
            "Add the Matrix Dataset to this Kaggle notebook."
        )

    print("Dataset files detected:")
    for path in data_files:
        print(f"  {path} ({path.stat().st_size / (1024 ** 2):.2f} MB)")

    return data_files


def read_signature(path, n=16):
    """Read the first bytes so we can diagnose mislabeled or compressed files."""
    with open(path, "rb") as f:
        return f.read(n)


def load_data_object(path):
    """Try several safe readers for Kaggle files that may be mislabeled as .pkl."""
    signature = read_signature(path)
    print(f"First 16 bytes: {signature.hex(' ')}")

    loaders = [
        ("pickle.load", lambda p: pickle.load(open(p, "rb"))),
        ("pandas.read_pickle", lambda p: pd.read_pickle(p)),
        ("pandas.read_pickle gzip", lambda p: pd.read_pickle(p, compression="gzip")),
        ("pandas.read_pickle bz2", lambda p: pd.read_pickle(p, compression="bz2")),
        ("pandas.read_pickle xz", lambda p: pd.read_pickle(p, compression="xz")),
        ("pandas.read_pickle zip", lambda p: pd.read_pickle(p, compression="zip")),
        ("pandas.read_pickle zstd", lambda p: pd.read_pickle(p, compression="zstd")),
        ("numpy.load allow_pickle", lambda p: np.load(p, allow_pickle=True)),
        ("pandas.read_parquet", lambda p: pd.read_parquet(p)),
        ("pandas.read_feather", lambda p: pd.read_feather(p)),
        ("pandas.read_hdf", lambda p: pd.read_hdf(p)),
    ]

    try:
        import joblib
        loaders.insert(2, ("joblib.load", lambda p: joblib.load(p)))
    except Exception:
        pass

    errors = []
    for label, loader in loaders:
        try:
            print(f"Trying loader: {label}")
            obj = loader(path)
            print(f"Loaded successfully with: {label}")
            describe_object(obj)
            return obj
        except Exception as exc:
            errors.append(f"{label}: {type(exc).__name__}: {exc}")

    short_errors = "\n".join("  - " + msg for msg in errors[:8])
    raise RuntimeError(
        "Could not load the Matrix Dataset file with the supported readers.\n"
        f"File: {path}\n"
        f"First 16 bytes: {signature.hex(' ')}\n"
        f"First loader errors:\n{short_errors}"
    )


def describe_object(obj):
    """Print a compact summary of the loaded dataset object for Kaggle output."""
    print(f"Loaded object type: {type(obj)}")

    if isinstance(obj, pd.DataFrame):
        print(f"DataFrame shape: {obj.shape}")
        print("First columns:", list(obj.columns[:10]))
        print("Dtype counts:")
        print(obj.dtypes.astype(str).value_counts())
        if len(obj) > 0 and len(obj.columns) > 0:
            sample = obj.iloc[0, : min(5, len(obj.columns))]
            print("First-row sample value types:", [type(v).__name__ for v in sample.to_list()])
        return

    if isinstance(obj, pd.Series):
        print(f"Series shape: {obj.shape}, dtype={obj.dtype}")
        if len(obj) > 0:
            print(f"First value type: {type(obj.iloc[0]).__name__}")
        return

    if isinstance(obj, np.ndarray):
        print(f"Array shape: {obj.shape}, dtype={obj.dtype}")
        return

    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"Dictionary keys sample: {keys[:10]}")
        return

    if isinstance(obj, (list, tuple)):
        print(f"Sequence length: {len(obj)}")
        if len(obj) > 0:
            print(f"First item type: {type(obj[0]).__name__}")


def normalize_matrix(candidate, matrix_size=MATRIX_SIZE):
    """Convert a possible matrix object to a 1024 x 1024 float32 NumPy array."""
    if isinstance(candidate, pd.DataFrame):
        candidate = candidate.to_numpy()
    elif isinstance(candidate, pd.Series):
        candidate = candidate.to_numpy()

    arr = np.asarray(candidate)
    arr = np.squeeze(arr)

    # Exact 2D matrix case.
    if arr.ndim == 2:
        if arr.shape == (matrix_size, matrix_size):
            return np.ascontiguousarray(arr, dtype=np.float32)
        if arr.size == matrix_size * matrix_size:
            return np.ascontiguousarray(arr.reshape(matrix_size, matrix_size), dtype=np.float32)

    # Some files store one matrix as a flattened vector.
    if arr.ndim == 1 and arr.size == matrix_size * matrix_size:
        return np.ascontiguousarray(arr.reshape(matrix_size, matrix_size), dtype=np.float32)

    return None


def append_if_matrix(candidate, matrices, limit=2):
    """Append candidate if it can be interpreted as one 1024 x 1024 matrix."""
    if len(matrices) >= limit:
        return True

    matrix = normalize_matrix(candidate)
    if matrix is not None:
        matrices.append(matrix)
        return True

    return False


def append_numeric_values(values, parts, needed_values):
    """Append finite numeric values until enough values exist for two matrices."""
    if sum(part.size for part in parts) >= needed_values:
        return

    try:
        arr = np.asarray(values, dtype=np.float32).ravel()
    except Exception:
        return

    if arr.size == 0:
        return

    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return

    current = sum(part.size for part in parts)
    remaining = needed_values - current
    parts.append(arr[:remaining].copy())


def matrices_from_flat_values(parts):
    """Build A and B from collected flat numeric values."""
    if not parts:
        return None

    flat = np.concatenate(parts)
    if flat.size < VALUES_PER_MATRIX:
        return None

    A = np.ascontiguousarray(flat[:VALUES_PER_MATRIX].reshape(MATRIX_SIZE, MATRIX_SIZE), dtype=np.float32)

    if flat.size >= VALUES_FOR_TWO_MATRICES:
        B = np.ascontiguousarray(
            flat[VALUES_PER_MATRIX:VALUES_FOR_TWO_MATRICES].reshape(MATRIX_SIZE, MATRIX_SIZE),
            dtype=np.float32,
        )
    else:
        print("Only enough numeric values for one matrix were found. Using B = A copy.")
        B = A.copy()

    return A, B


def collect_numeric_values_from_dataframe(df, needed_values=VALUES_FOR_TWO_MATRICES):
    """Collect enough numeric DataFrame values to form two 1024 x 1024 matrices."""
    parts = []

    numeric_columns = list(df.select_dtypes(include=[np.number]).columns)
    print(f"Numeric DataFrame columns detected: {len(numeric_columns)}")

    # Process one column at a time to avoid copying the full 3.7 GB table.
    for col in numeric_columns:
        values = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float32, copy=True)
        append_numeric_values(values, parts, needed_values)
        if sum(part.size for part in parts) >= needed_values:
            return matrices_from_flat_values(parts)

    # Some tables store numbers as strings or arrays in object columns.
    object_columns = list(df.select_dtypes(include=["object"]).columns)
    print(f"Object DataFrame columns detected: {len(object_columns)}")

    for col in object_columns:
        series = df[col]

        # First try numeric strings in the whole column.
        numeric_series = pd.to_numeric(series, errors="coerce")
        if numeric_series.notna().any():
            append_numeric_values(numeric_series.to_numpy(dtype=np.float32, copy=True), parts, needed_values)
            if sum(part.size for part in parts) >= needed_values:
                return matrices_from_flat_values(parts)

        # Then inspect object values that may be arrays/lists.
        for value in series.head(5000):
            if isinstance(value, (np.ndarray, list, tuple, pd.Series, pd.DataFrame)):
                matrix = normalize_matrix(value)
                if matrix is not None:
                    append_numeric_values(matrix.ravel(), parts, needed_values)
                else:
                    try:
                        append_numeric_values(np.asarray(value).ravel(), parts, needed_values)
                    except Exception:
                        pass

            if sum(part.size for part in parts) >= needed_values:
                return matrices_from_flat_values(parts)

    return matrices_from_flat_values(parts)


def collect_from_dataframe(df, matrices, limit=2):
    """Extract matrices from numeric or object DataFrame layouts."""
    # Layout 1: the DataFrame itself is exactly one matrix.
    if df.shape == (MATRIX_SIZE, MATRIX_SIZE):
        try:
            matrices.append(np.ascontiguousarray(df.to_numpy(dtype=np.float32, copy=True), dtype=np.float32))
            return
        except Exception:
            pass

    numeric_df = df.select_dtypes(include=[np.number])

    # Layout 2: one large numeric table where each 1024-row or 1024-column block is a matrix.
    if numeric_df.shape[0] >= MATRIX_SIZE and numeric_df.shape[1] >= MATRIX_SIZE:
        matrices.append(
            np.ascontiguousarray(
                numeric_df.iloc[:MATRIX_SIZE, :MATRIX_SIZE].to_numpy(dtype=np.float32, copy=True)
            )
        )
        if len(matrices) < limit:
            if numeric_df.shape[0] >= MATRIX_SIZE * 2:
                matrices.append(
                    np.ascontiguousarray(
                        numeric_df.iloc[MATRIX_SIZE:MATRIX_SIZE * 2, :MATRIX_SIZE].to_numpy(
                            dtype=np.float32,
                            copy=True,
                        )
                    )
                )
            elif numeric_df.shape[1] >= MATRIX_SIZE * 2:
                matrices.append(
                    np.ascontiguousarray(
                        numeric_df.iloc[:MATRIX_SIZE, MATRIX_SIZE:MATRIX_SIZE * 2].to_numpy(
                            dtype=np.float32,
                            copy=True,
                        )
                    )
                )
        return

    # Layout 3: result-table format. Use the first enough numeric values to form A and B.
    built = collect_numeric_values_from_dataframe(df)
    if built is not None:
        A, B = built
        matrices.append(A)
        if len(matrices) < limit:
            matrices.append(B)
        return

    # Layout 4: object cells contain NumPy arrays/lists. Inspect a sample.
    object_df = df.select_dtypes(include=["object"]).head(5000)
    for value in object_df.to_numpy().ravel():
        collect_matrices(value, matrices, limit)
        if len(matrices) >= limit:
            return


def collect_from_array(arr, matrices, limit=2):
    """Extract matrices from NumPy arrays, including stacked matrices."""
    if len(matrices) >= limit:
        return

    arr = np.asarray(arr)

    if arr.ndim == 3:
        # Common layout: (number_of_matrices, 1024, 1024).
        if arr.shape[1:] == (MATRIX_SIZE, MATRIX_SIZE):
            for i in range(arr.shape[0]):
                matrices.append(np.ascontiguousarray(arr[i], dtype=np.float32))
                if len(matrices) >= limit:
                    return

        # Alternative layout: (1024, 1024, number_of_matrices).
        if arr.shape[:2] == (MATRIX_SIZE, MATRIX_SIZE):
            for i in range(arr.shape[2]):
                matrices.append(np.ascontiguousarray(arr[:, :, i], dtype=np.float32))
                if len(matrices) >= limit:
                    return

    if arr.ndim == 2:
        if append_if_matrix(arr, matrices, limit):
            return

        # Vertically stacked matrices: (k * 1024, 1024).
        if arr.shape[1] >= MATRIX_SIZE and arr.shape[0] >= MATRIX_SIZE:
            for start in range(0, arr.shape[0] - MATRIX_SIZE + 1, MATRIX_SIZE):
                matrices.append(np.ascontiguousarray(arr[start:start + MATRIX_SIZE, :MATRIX_SIZE], dtype=np.float32))
                if len(matrices) >= limit:
                    return

        # Horizontally stacked matrices: (1024, k * 1024).
        if arr.shape[0] >= MATRIX_SIZE and arr.shape[1] >= MATRIX_SIZE:
            for start in range(0, arr.shape[1] - MATRIX_SIZE + 1, MATRIX_SIZE):
                matrices.append(np.ascontiguousarray(arr[:MATRIX_SIZE, start:start + MATRIX_SIZE], dtype=np.float32))
                if len(matrices) >= limit:
                    return

    # Flat numeric array with enough values for one or two matrices.
    if arr.ndim == 1 and np.issubdtype(arr.dtype, np.number):
        built = matrices_from_flat_values([arr.astype(np.float32, copy=False)])
        if built is not None:
            A, B = built
            matrices.append(A)
            if len(matrices) < limit:
                matrices.append(B)
            return

    if arr.dtype == object and arr.size <= 10000:
        for value in arr.ravel()[:5000]:
            collect_matrices(value, matrices, limit)
            if len(matrices) >= limit:
                return

    append_if_matrix(arr, matrices, limit)


def collect_matrices(obj, matrices, limit=2):
    """Recursively collect matrices from arrays, DataFrames, lists, dictionaries, and npz files."""
    if len(matrices) >= limit:
        return

    if isinstance(obj, np.lib.npyio.NpzFile):
        for key in obj.files:
            collect_matrices(obj[key], matrices, limit)
            if len(matrices) >= limit:
                return
        return

    if isinstance(obj, pd.DataFrame):
        collect_from_dataframe(obj, matrices, limit)
        return

    if isinstance(obj, pd.Series):
        if append_if_matrix(obj, matrices, limit):
            return
        built = matrices_from_flat_values([pd.to_numeric(obj, errors="coerce").dropna().to_numpy(dtype=np.float32)])
        if built is not None:
            A, B = built
            matrices.append(A)
            if len(matrices) < limit:
                matrices.append(B)
            return
        for value in obj.head(5000):
            collect_matrices(value, matrices, limit)
            if len(matrices) >= limit:
                return
        return

    if isinstance(obj, dict):
        for value in obj.values():
            collect_matrices(value, matrices, limit)
            if len(matrices) >= limit:
                return
        return

    if isinstance(obj, (list, tuple)):
        for value in obj:
            collect_matrices(value, matrices, limit)
            if len(matrices) >= limit:
                return
        return

    collect_from_array(obj, matrices, limit)


def matrices_from_dataset_bytes(path):
    """Last fallback: derive benchmark matrices from the Kaggle file bytes, with no download."""
    print("No direct matrix/table values were extracted. Deriving benchmark matrices from dataset file bytes.")
    needed_uint16 = VALUES_FOR_TWO_MATRICES
    needed_bytes = needed_uint16 * np.dtype(np.uint16).itemsize

    with open(path, "rb") as f:
        raw = f.read(needed_bytes)

    data = np.frombuffer(raw, dtype=np.uint16).astype(np.float32)
    if data.size < VALUES_PER_MATRIX:
        raise ValueError("The dataset file is too small to derive a 1024 x 1024 matrix.")

    # Scale uint16 values to a floating-point range useful for matrix addition.
    data /= np.float32(np.iinfo(np.uint16).max)

    A = np.ascontiguousarray(data[:VALUES_PER_MATRIX].reshape(MATRIX_SIZE, MATRIX_SIZE), dtype=np.float32)
    if data.size >= VALUES_FOR_TWO_MATRICES:
        B = np.ascontiguousarray(data[VALUES_PER_MATRIX:VALUES_FOR_TWO_MATRICES].reshape(MATRIX_SIZE, MATRIX_SIZE), dtype=np.float32)
    else:
        B = A.copy()

    return A, B


def load_two_matrices_from_kaggle():
    """Load matrices A and B from Kaggle-mounted Matrix Dataset files."""
    matrices = []
    last_error = None
    last_path = None

    for data_path in find_matrix_data_files():
        last_path = data_path
        print(f"Loading: {data_path}")
        try:
            obj = load_data_object(data_path)
        except Exception as exc:
            last_error = exc
            print(f"Could not load {data_path}: {type(exc).__name__}: {exc}")
            continue

        collect_matrices(obj, matrices, limit=2)
        if len(matrices) >= 2:
            break

    if len(matrices) == 0:
        if last_path is not None:
            A, B = matrices_from_dataset_bytes(last_path)
            print(f"Matrix A shape={A.shape}, dtype={A.dtype}")
            print(f"Matrix B shape={B.shape}, dtype={B.dtype}")
            return A, B
        if last_error is not None:
            raise RuntimeError(f"No matrix could be loaded. Last loading error:\n{last_error}")
        raise ValueError("No 1024 x 1024 matrix could be extracted from the Kaggle dataset files.")

    if len(matrices) == 1:
        # Fallback keeps the notebook runnable if only one matrix is mounted.
        print("Warning: only one matrix was found. Using B = A copy for demonstration.")
        matrices.append(matrices[0].copy())

    A, B = matrices[0], matrices[1]
    print(f"Matrix A shape={A.shape}, dtype={A.dtype}")
    print(f"Matrix B shape={B.shape}, dtype={B.dtype}")
    return A, B


# Stop early if the Kaggle notebook is not using a GPU.
if not cuda.is_available():
    raise RuntimeError("CUDA is not available. In Kaggle, enable Settings -> Accelerator -> GPU.")

device = cuda.get_current_device()
print(f"CUDA device: {device.name.decode() if isinstance(device.name, bytes) else device.name}")
WORKING_ROOT.mkdir(parents=True, exist_ok=True)

CUDA device: Tesla T4


In [8]:
# A 16 x 16 CUDA block has 256 threads, a common choice for 2D matrix kernels.
THREADS_PER_BLOCK = (16, 16)


@cuda.jit
def matrix_add_kernel(A, B, C, rows, cols):
    """Each CUDA thread computes one element: C[row, col] = A[row, col] + B[row, col]."""
    col, row = cuda.grid(2)

    # Boundary check is required because the grid can be slightly larger than the matrix.
    if row < rows and col < cols:
        C[row, col] = A[row, col] + B[row, col]


def cuda_matrix_add(A, B):
    """Transfer A and B to GPU, run the addition kernel, and return C on CPU."""
    rows, cols = A.shape
    C = np.empty_like(A, dtype=np.float32)

    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array_like(d_A)

    blocks_per_grid_x = (cols + THREADS_PER_BLOCK[0] - 1) // THREADS_PER_BLOCK[0]
    blocks_per_grid_y = (rows + THREADS_PER_BLOCK[1] - 1) // THREADS_PER_BLOCK[1]
    blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

    start = time.perf_counter()
    matrix_add_kernel[blocks_per_grid, THREADS_PER_BLOCK](d_A, d_B, d_C, rows, cols)
    cuda.synchronize()
    gpu_elapsed = time.perf_counter() - start

    d_C.copy_to_host(C)
    return C, gpu_elapsed, blocks_per_grid

In [9]:
# Load two 1024 x 1024 matrices from the Kaggle Matrix Dataset pickle files.
A, B = load_two_matrices_from_kaggle()

# Run CUDA matrix addition.
C, gpu_elapsed, blocks_per_grid = cuda_matrix_add(A, B)

# CPU result is used only to verify CUDA correctness.
cpu_reference = A + B
max_abs_error = float(np.max(np.abs(C - cpu_reference)))

# Save only a small preview so the Kaggle output remains lightweight.
sample_path = WORKING_ROOT / "ds2_matrix_addition_cuda_sample.csv"
pd.DataFrame(C[:10, :10]).to_csv(sample_path, index=False)

print("\nDS2 Matrix Addition complete.")
print(f"Input shape: {A.shape}")
print(f"CUDA blocks per grid: {blocks_per_grid}")
print(f"Threads per block: {THREADS_PER_BLOCK}")
print(f"Total elements processed: {C.size:,}")
print(f"Max absolute error vs CPU: {max_abs_error:.3e}")
print(f"Kernel elapsed time including synchronization: {gpu_elapsed:.6f} seconds")
print(f"10 x 10 result preview saved to: {sample_path}")
print("Top-left 5 x 5 of C:")
print(C[:5, :5])

Dataset files detected:
  /kaggle/input/datasets/julianezrasamuel/matrix-dataset/results_dataframe.pkl (3732.87 MB)
Loading: /kaggle/input/datasets/julianezrasamuel/matrix-dataset/results_dataframe.pkl
First 16 bytes: 80 04 95 64 01 00 00 00 00 00 00 8c 11 70 61 6e
Trying loader: pickle.load
Trying loader: pandas.read_pickle
Trying loader: joblib.load
Loaded successfully with: joblib.load
Loaded object type: <class 'pandas.core.frame.DataFrame'>
DataFrame shape: (10980480, 5)
First columns: ['n', 'k', 'm', 'result', 'P']
Dtype counts:
int64      3
float64    1
object     1
Name: count, dtype: int64
First-row sample value types: ['int64', 'int64', 'int64', 'float64', 'ndarray']
Numeric DataFrame columns detected: 4
Matrix A shape=(1024, 1024), dtype=float32
Matrix B shape=(1024, 1024), dtype=float32

DS2 Matrix Addition complete.
Input shape: (1024, 1024)
CUDA blocks per grid: (64, 64)
Threads per block: (16, 16)
Total elements processed: 1,048,576
Max absolute error vs CPU: 0.000e+00
K